# Validation des solveurs — SOFA vs FreeFem++ vs analytique

## Objectif

Valider le solveur éléments finis de SOFA (`LinearSmallStrainFEMForceField`) en le comparant,
cas par cas, à FreeFem++ (référence numérique indépendante) et, quand elle existe, à la
solution analytique (référence exacte).

## Organisation du projet

```
projet_validation/
├── notebook_validation.ipynb   <- ce fichier
├── common/                     <- métriques et tracés génériques
└── cases/
    ├── bar_1d/                 <- barre 1D
    ├── case_2d/                <- distributed_load, compression,
    │                              three_point_bending
    └── case_3d/                <- distributed_load, three_point_bending,
                                   circular_traction, circular_bending, torsion
```

Chaque cas expose une seule fonction `run_case()` qui encapsule tout le détail
(paramètres, appel FreeFem, appel SOFA, calcul analytique). Le notebook ne fait
qu'importer et appeler — aucun détail d'implémentation ne traîne ici.

## Imports — tous les cas et utilitaires, en un seul endroit

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110

from common.metrics import compare_and_report, compare_far_field, report_eb_check, report_midspan_deflection, report_relative_error, report_torsion_angle
from common.plotting import plot_displacement, plot_geometry, plot_fields_2d, plot_parity

from cases.bar_1d.run import run_case as run_bar_1d, MATH_DESCRIPTION as MATH_1D
from cases.case_2d.distributed_load.run import run_case as run_2d_dist, MATH_DESCRIPTION as MATH_2D_DIST
from cases.case_2d.compression.run import run_case as run_2d_compression, MATH_DESCRIPTION as MATH_2D_COMPRESSION
from cases.case_2d.three_point_bending.run import run_case as run_3pt_bending, MATH_DESCRIPTION as MATH_2D_3PT
from cases.case_3d.distributed_load.run import run_case as run_3d_dist, MATH_DESCRIPTION as MATH_3D_DIST
from cases.case_3d.three_point_bending.run import run_case as run_3d_3pt, MATH_DESCRIPTION as MATH_3D_3PT
from cases.case_3d.circular_traction.run import run_case as run_3d_circular, MATH_DESCRIPTION as MATH_3D_CIRCULAR
from cases.case_3d.circular_bending.run import run_case as run_3d_circular_bending, MATH_DESCRIPTION as MATH_3D_CIRCULAR_BENDING
from cases.case_3d.torsion.run import run_case as run_3d_torsion, MATH_DESCRIPTION as MATH_3D_TORSION


---
## Cas 1 — Barre 1D, charge répartie

In [ ]:
print(MATH_1D)


In [ ]:
results_1d = run_bar_1d()
compare_and_report(results_1d)


In [ ]:
plot_displacement(results_1d)
plot_geometry(results_1d)


---
## Cas 2 — Poutre 2D, charge distribuée (plane strain)

Régime déformations planes (`mode="plane_strain"`, le défaut). Le régime contraintes planes fait l'objet du Cas 3.

In [ ]:
print(MATH_2D_DIST)


In [ ]:
results_2d_dist = run_2d_dist(mode="plane_strain")
compare_and_report(results_2d_dist)


Pas de solution analytique pour ce cas : la validation se fait uniquement par cross-comparaison SOFA vs FreeFem (RMS global + RMS par composante ux/uy, et visuel du champ).

In [ ]:
plot_fields_2d(results_2d_dist)


---
## Cas 3 — Poutre 2D, charge distribuée (plane stress)

Même chargement, même maillage et mêmes E, ν que le Cas 2 : seul le régime
change. En contraintes planes, SOFA utilise `template="Vec2d"` (élasticité 2D
genuine, aucun ddl z) et FreeFem prend λ = E·ν/(1−ν²), au lieu de `Vec3d` avec
z bloqué et λ = E·ν/((1+ν)(1−2ν)) en déformations planes.

In [ ]:
results_2d_dist_cp = run_2d_dist(mode="plane_stress")
compare_and_report(results_2d_dist_cp)


Comme en déformations planes, ce cas n'a pas de solution analytique : la validation se fait par cross-comparaison SOFA vs FreeFem (RMS global + RMS par composante ux/uy, et visuel du champ).

In [ ]:
plot_fields_2d(results_2d_dist_cp)


---
## Cas 4 — Poutre 2D, compression (traction en bord droit, plane strain)

Régime déformations planes (`mode="plane_strain"`, le défaut). Le régime contraintes planes fait l'objet du Cas 5.

In [ ]:
print(MATH_2D_COMPRESSION)


In [ ]:
results_2d_compression = run_2d_compression(mode="plane_strain")
compare_and_report(results_2d_compression)


En plus du RMS SOFA vs FreeFem (valable partout), une vérification analytique en champ lointain est possible : la solution exacte de compression uniforme n'est valable qu'à distance du bord encastré (effet Saint-Venant, seuil marqué en pointillés sur les graphiques ci-dessous).

In [ ]:
compare_far_field(results_2d_compression)


In [ ]:
plot_fields_2d(results_2d_compression)


---
## Cas 5 — Poutre 2D, compression (plane stress)

Même chargement que le Cas 4, en contraintes planes. Ici le mode change aussi
la référence analytique en champ lointain — `run_case` sélectionne la formule
correspondante :

    plane strain : ux = −q(1−ν²)/E · x    uy = q·ν(1+ν)/E · y
    plane stress : ux = −q/E · x          uy = q·ν/E · y

In [ ]:
results_2d_compression_cp = run_2d_compression(mode="plane_stress")
compare_and_report(results_2d_compression_cp)


Le seuil de Saint-Venant est le même qu'au Cas 4 (x ≥ 2·H) : seule la formule comparée change.

In [ ]:
compare_far_field(results_2d_compression_cp)


In [ ]:
plot_fields_2d(results_2d_compression_cp)


---
## Cas 6 — Poutre 2D, flexion 3 points

Le mode se choisit à l'appel de `run_3pt_bending(mode=...)` : `"plane_strain"` (défaut) ou `"plane_stress"`.

In [ ]:
print(MATH_2D_3PT)


In [ ]:
results_3pt = run_3pt_bending(mode="plane_strain")   # ou mode="plane_stress"
compare_and_report(results_3pt)


Une vérification Euler-Bernoulli de la flèche à mi-portée est fournie **à titre indicatif seulement** — la poutre est courte et épaisse (L/H = 5), donc un écart avec la théorie de poutre 1D est attendu, pas un signe d'erreur.

In [ ]:
report_eb_check(results_3pt)


In [ ]:
plot_fields_2d(results_3pt)


---
## Cas 7 — Poutre 3D, charge distribuée (face supérieure)

In [ ]:
print(MATH_3D_DIST)


In [ ]:
results_3d_dist = run_3d_dist()
compare_and_report(results_3d_dist)


Pas de solution analytique pour ce cas : cross-validation SOFA vs FreeFem uniquement. En 3D, une carte de couleur sur le maillage (tripcolor) ne s'applique pas directement (tétraèdres) — on utilise un graphique de parité à la place (points proches de la diagonale = bon accord).

In [ ]:
plot_parity(results_3d_dist)


---
## Cas 8 — Poutre 3D, flexion 3 points

In [ ]:
print(MATH_3D_3PT)


In [ ]:
results_3d_3pt = run_3d_3pt()
compare_and_report(results_3d_3pt)


In [ ]:
report_midspan_deflection(results_3d_3pt)


In [ ]:
plot_parity(results_3d_3pt)


---
## Cas 9 — Poutre 3D circulaire, traction axiale

In [ ]:
print(MATH_3D_CIRCULAR)


In [ ]:
results_3d_circular = run_3d_circular()
compare_and_report(results_3d_circular)


Vérification analytique en champ lointain (traction uniaxiale simple, valable pour x >= exclusionFactor·r). Ici pas de recentrage de moyenne nécessaire : la face x=0 est entièrement encastrée (ux=uy=uz=0 sur toute la face), donc aucune translation/rotation rigide résiduelle contrairement au cas compression 2D.

In [ ]:
compare_far_field(results_3d_circular, mean_shift_components=())


In [ ]:
plot_parity(results_3d_circular)


---
## Cas 10 — Poutre 3D circulaire, flexion (charge transverse vers le bas)

In [ ]:
print(MATH_3D_CIRCULAR_BENDING)


In [ ]:
results_3d_circular_bending = run_3d_circular_bending()
compare_and_report(results_3d_circular_bending)


Pas de référence analytique pour ce cas (cf. MATH_DESCRIPTION) : cross-validation SOFA vs FreeFem uniquement.

In [ ]:
plot_parity(results_3d_circular_bending)


---
## Cas 11 — Poutre 3D circulaire, torsion pure (dernier cas)

In [ ]:
print(MATH_3D_TORSION)


In [ ]:
results_3d_torsion = run_3d_torsion()
compare_and_report(results_3d_torsion)


Contrairement aux autres cas 3D, la solution analytique est ici **exacte sur tout le domaine** (une section circulaire ne se gauchit pas sous torsion pure — pas de couche limite de Saint-Venant à exclure).

In [ ]:
report_relative_error(results_3d_torsion)


In [ ]:
report_torsion_angle(results_3d_torsion)


In [ ]:
plot_parity(results_3d_torsion)


**Écart visible sur ce cas** : contrairement aux cas de traction/flexion 3D, l'écart entre SOFA/FreeFem et l'analytique reste ici franchement perceptible sur le graphique de parité (points nettement décalés sous la diagonale pour uy/uz), pas seulement dans les RMS. Ce n'est pas un bug de script ni des formules `T/J` — la cause identifiée est le **locking numérique des tétraèdres P1** en régime de flexion/torsion, combiné à un **maillage encore trop grossier** pour ce mode de déformation (une étude de convergence sur le cas rectangulaire équivalent donne un ordre effectif ~0.26, très en dessous du O(h) attendu, et l'écart se réduit franchement en raffinant : 43%→31%→21%→17% de 105 à 1889 noeuds). Le cas de traction axiale pure n'est pas concerné par ce mode de locking, d'où l'accord quasi-exact obtenu au Cas 9.

Piste d'action : raffiner le maillage circulaire (voir `mesh_size` dans `cases/case_3d/torsion/params_mesh.json`) et refaire cette cellule pour vérifier que l'écart SOFA/FreeFem/analytique diminue de façon monotone.

---
## Conclusion

- Les RMS et les visuels ci-dessus mesurent l'écart entre SOFA, FreeFem et (quand elle existe) l'analytique, pour chaque cas.
- Ce format (un module par cas + notebook qui n'orchestre que l'appel) permet d'ajouter de nouveaux cas de validation sans complexifier le notebook lui-même.
- Campagne de validation complète, 11 cas : 1D (barre) ; 2D (charge distribuée et compression, chacune en déformations planes puis en contraintes planes ; flexion 3 points) ; 3D (charge distribuée, flexion 3 points, traction circulaire, flexion circulaire, torsion).
- Les deux régimes 2D passent par le même module de cas via `run_case(mode=...)` : une seule scène SOFA et un seul `.edp` par cas, pas de duplication entre plane strain et plane stress.
